# Unified pipeline: local + Colab (save artifacts, upload to Hugging Face & GitHub)

This notebook works on **both local machines and Colab**:
- **Local**: reads secrets from `.env` file or environment variables, saves artifacts locally
- **Colab**: mounts Google Drive, reads secrets from env/getpass, saves to Drive

Secrets are read from (in order of priority):
1. Environment variables (`HF_TOKEN`, `GH_TOKEN`, `HF_REPO_ID`)
2. Local `.env` file (if running locally)
3. Interactive prompt via `getpass` (secure, not stored)

The notebook does NOT auto-push to GitHub; it prepares commits and shows the commands.

**Setup (local machine only):**
Create a `.env` file in your project root:
```
HF_TOKEN=hf_your_token_here
GH_TOKEN=ghp_your_github_token_here
HF_REPO_ID=your-username/your-repo
```

In [ ]:
import os
import sys
from pathlib import Path
from getpass import getpass

# Detect if running in Colab
IS_COLAB = 'google.colab' in sys.modules
print(f'Running in: {"Colab" if IS_COLAB else "Local environment"}')

# Set artifact base directory
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ARTIFACT_BASE = '/content/drive/MyDrive/colab_experiments/sub1quant'
else:
    # Local: use a workspace-relative directory
    ARTIFACT_BASE = str(Path.cwd() / 'artifacts')

os.makedirs(ARTIFACT_BASE, exist_ok=True)
print(f'Artifact base directory: {ARTIFACT_BASE}')

In [ ]:
# Install dependencies (run once per session)
import subprocess
import sys
packages = ['huggingface_hub', 'gitpython', 'torch', 'torchvision', 'python-dotenv']
for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg, '--upgrade'])
print('Installed core packages:', ', '.join(packages))

## Read secrets (token) securely
The cells below try to read tokens from environment variables first. If not present, you will be prompted to paste them securely using `getpass`.

Set environment variables in Colab before running a cell, for example:
`import os; os.environ['HF_TOKEN'] = 'hf_...'` (temporary for the session)

In [ ]:
# Load secrets from .env file (local) or environment variables
import os
from pathlib import Path
from getpass import getpass

def load_secrets():
    """Load HF_TOKEN, GH_TOKEN, HF_REPO_ID from env or .env file."""
    secrets = {}
    env_file = Path('.env')
    
    # Try to load from .env if it exists
    if env_file.exists():
        print(f'Loading secrets from {env_file}')
        with open(env_file) as f:
            for line in f:
                line = line.strip()
                if line and not line.startswith('#'):
                    if '=' in line:
                        key, val = line.split('=', 1)
                        secrets[key.strip()] = val.strip().strip('"\'')
    
    # Environment variables override .env
    for key in ['HF_TOKEN', 'GH_TOKEN', 'HF_REPO_ID']:
        if key in os.environ:
            secrets[key] = os.environ[key]
    
    return secrets

secrets = load_secrets()
HF_TOKEN = secrets.get('HF_TOKEN')
GH_TOKEN = secrets.get('GH_TOKEN')
HF_REPO_ID = secrets.get('HF_REPO_ID')

# Prompt for missing secrets
if not HF_TOKEN:
    HF_TOKEN = getpass('Hugging Face token (input hidden, not stored): ')

if not HF_REPO_ID:
    HF_REPO_ID = input('Hugging Face repo ID (format: username/repo): ')

if not GH_TOKEN:
    print('GH_TOKEN not set; GitHub push commands will be prepared but will not run.')

print(f'Secrets loaded: HF_TOKEN={bool(HF_TOKEN)}, GH_TOKEN={bool(GH_TOKEN)}, HF_REPO_ID={HF_REPO_ID}')

## Example: create and save tensors / model checkpoints to Drive

In [ ]:
import json
import os

# Example: save experiment metadata and intermediate results
# Customize for your quantization/training metrics
results = {
    'experiment': 'sub1bit_hybrid_ics_svd',
    'model': 'gemma-2b',
    'layer': 0,
    'bits_per_weight': 0.85,
    'compression_ratio': 15.2,
    'rmse': 0.042,
    'timestamp': '2026-06-21T12:00:00Z'
}
results_path = os.path.join(ARTIFACT_BASE, 'experiment_results.json')
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'Saved experiment results to {results_path}')

In [ ]:
import torch
import os

# Example: create a demo tensor
# Customize this cell for your actual experiment (e.g., load model weights, quantization results, etc.)
t = torch.randn(1024, 1024, dtype=torch.float32)
ckpt_path = os.path.join(ARTIFACT_BASE, 'demo_tensor.pt')
torch.save({'tensor': t, 'metadata': {'shape': list(t.shape), 'dtype': str(t.dtype)}}, ckpt_path)
print(f'Saved demo tensor to {ckpt_path}')
print(f'File size: {os.path.getsize(ckpt_path) / 1e6:.2f} MB')

## Upload artifacts to Hugging Face Hub

In [ ]:
from huggingface_hub import HfApi, upload_file
import os

if not HF_TOKEN:
    print('HF_TOKEN not set; skipping upload.')
else:
    api = HfApi()
    
    # Upload demo tensor
    local_file = ckpt_path
    path_in_repo = 'artifacts/demo_tensor.pt'
    print(f'Uploading {local_file} to {HF_REPO_ID}/{path_in_repo} ...')
    try:
        upload_file(
            path_or_fileobj=local_file,
            path_in_repo=path_in_repo,
            repo_id=HF_REPO_ID,
            token=HF_TOKEN,
            commit_message='Add demo tensor from experiment'
        )
        print('Upload complete')
    except Exception as e:
        print(f'Upload failed: {e}')
        print('Ensure HF_REPO_ID is correct and token has write access.')

## Prepare a Git commit for GitHub (manual push)
The next cell initializes a local git repo in Drive and commits selected artifacts. It will NOT push. If you want to push from Colab, set `GH_TOKEN` and run the provided push commands yourself.

Note: automatic pushing may expose your token in the notebook's command history; prefer running push from your local machine using your SSH key or personal token with caution.

In [ ]:
from git import Repo
import os
from pathlib import Path
import shutil

# Create a git repo in the artifacts directory
repo_dir = os.path.join(ARTIFACT_BASE, 'git_repo')
os.makedirs(repo_dir, exist_ok=True)

if not os.path.exists(os.path.join(repo_dir, '.git')):
    repo = Repo.init(repo_dir)
    print(f'Initialized git repo at {repo_dir}')
else:
    repo = Repo(repo_dir)
    print(f'Using existing repo at {repo_dir}')

# Copy artifacts into the repo
dest_tensor = os.path.join(repo_dir, 'demo_tensor.pt')
dest_results = os.path.join(repo_dir, 'experiment_results.json')
shutil.copy(ckpt_path, dest_tensor)
shutil.copy(results_path, dest_results)

# Stage and commit
repo.index.add(['demo_tensor.pt', 'experiment_results.json'])
repo.index.commit('Add artifacts from experiment')
print(f'Committed artifacts to {repo_dir}')
print(f'\nRepo status:')
print(repo.git.status())

print('\n=== GitHub Push Instructions ===')
print(f'Repo is ready at: {repo_dir}')
print()
print('Option 1 (SSH - recommended if you have SSH key set up):')
print('  cd ' + repo_dir)
print('  git remote add origin git@github.com:<YOUR_USER>/<YOUR_REPO>.git')
print('  git push origin main')
print()
print('Option 2 (HTTPS with GH_TOKEN):')
if GH_TOKEN:
    print(f'  cd {repo_dir}')
    print(f'  git remote add origin https://<YOUR_USER>:{GH_TOKEN}@github.com/<YOUR_USER>/<YOUR_REPO>.git')
    print('  git push origin main')
else:
    print('  (GH_TOKEN not set; set it in .env or environment and re-run)')
print()
print('Option 3 (Local push from your machine with SSH key):')
print('  1. Clone your GitHub repo locally')
print(f'  2. Copy {repo_dir}/* to the cloned folder')
print('  3. git add -A && git commit -m "Add artifacts" && git push')

## Summary and Best Practices

**What was done:**
- Detected local vs Colab environment
- Loaded secrets from `.env` file (local) or environment variables
- Saved tensors and experiment results to disk
- Uploaded artifacts to Hugging Face Hub
- Prepared a Git repo with artifacts (ready to push)

**Secrets Priority (in order):**
1. Environment variables (`HF_TOKEN`, `GH_TOKEN`, `HF_REPO_ID`)
2. `.env` file in project root (local only)
3. Interactive prompt via `getpass()` (secure, not stored)

**Next steps:**
1. Customize the artifact saving logic for your experiments
2. Set `HF_REPO_ID` to your actual Hugging Face repo
3. Push to GitHub using one of the methods above
4. For large files (>100MB), use Hugging Face's `git lfs` from your local machine

**Best practices:**
- Keep `.env` out of version control (add to `.gitignore`)
- Use environment variables in CI/CD pipelines
- Use SSH keys for GitHub instead of tokens when possible
- For local runs, create `.env` with content like:
  ```
  HF_TOKEN=hf_your_token
  GH_TOKEN=ghp_your_github_token
  HF_REPO_ID=your-username/your-repo
  ```